# FreeControl — e2e (structure fidelity vs prompt adherence + strength sweep)
Headline: keep the REFERENCE structure while CONTENT follows the prompt. Structure = depth-corr(result, ref)
(color-invariant); adherence = CLIP(result, target prompt). Sweep `structure_strength` (step cutoff) = the dial.
Runtime: 80GB A100.

In [ ]:
import subprocess
for _ in range(3):
    if subprocess.call(["pip","install","-q","git+https://github.com/huggingface/diffusers.git"])==0: break
!pip install -q transformers accelerate sentencepiece protobuf hf_transfer scikit-image

In [ ]:
import torch, os, numpy as np
os.environ["HF_HUB_ENABLE_HF_TRANSFER"]="1"
from huggingface_hub import login
try:
    from google.colab import userdata; login(userdata.get("HUGGINGFACE_TOKEN"))
except Exception:
    login()
DEV, DT = "cuda", torch.bfloat16
assert torch.cuda.is_available(); print("GPU:", torch.cuda.get_device_name(0))
REPO="remyxai/freecontrol-flux-modular"; H=W=1024

In [ ]:
from diffusers import ModularPipeline
pipe = ModularPipeline.from_pretrained(REPO, trust_remote_code=True)
assert type(pipe.blocks).__name__ == "FreeControlBlock", type(pipe.blocks).__name__
pipe.load_components(dtype=DT); pipe.to(DEV)   # base FLUX.1-dev ~36GB fits 80GB
def get(o):
    o = o.images if hasattr(o,"images") else (o.get("images") if isinstance(o,dict) else o)
    return o[0] if isinstance(o,(list,tuple)) else o
print("loaded:", type(pipe.blocks).__name__)

In [ ]:
from PIL import Image, ImageDraw
from skimage import data
from IPython.display import display
from transformers import pipeline as hf_pipeline, CLIPModel, CLIPProcessor
REF = Image.fromarray(data.astronaut()).convert("RGB").resize((W,H))
TARGET = "a bronze statue bust, museum, dramatic lighting"
_dep = hf_pipeline("depth-estimation", model="depth-anything/Depth-Anything-V2-Small-hf", device=0)
def dcorr(a,b):
    da=np.asarray(_dep(a)["depth"].convert("L").resize((256,256)),np.float32).ravel()
    db=np.asarray(_dep(b)["depth"].convert("L").resize((256,256)),np.float32).ravel()
    return float(np.corrcoef(da,db)[0,1])
_clip=CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(DEV).eval(); _cp=CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
@torch.no_grad()
def clip_prompt(im, txt):
    px=_cp(images=[im],return_tensors="pt").to(DEV); ti=_cp(text=[txt],return_tensors="pt",padding=True,truncation=True).to(DEV)
    o=_clip(pixel_values=px["pixel_values"],input_ids=ti["input_ids"],attention_mask=ti["attention_mask"])
    e=o.image_embeds/o.image_embeds.norm(dim=-1,keepdim=True); t=o.text_embeds/o.text_embeds.norm(dim=-1,keepdim=True)
    return float((e@t.T).squeeze().cpu())
print("metrics ready")

In [ ]:
def run(strength, ref=REF, seed=0):
    return get(pipe(prompt=TARGET, reference_image=ref, structure_strength=strength, num_inference_steps=28,
                    guidance_scale=6.5, generator=torch.Generator("cpu").manual_seed(seed),
                    height=H, width=W, output_type="pil", output="images"))
vanilla = get(pipe(prompt=TARGET, reference_image=None, num_inference_steps=28, guidance_scale=6.5,
                   generator=torch.Generator("cpu").manual_seed(0), height=H, width=W, output_type="pil", output="images"))
panels=[("reference",REF,None,None), ("no control",vanilla, dcorr(vanilla,REF), clip_prompt(vanilla,TARGET))]
for s in (0.2, 0.3, 0.5):
    im=run(s); panels.append((f"strength={s}", im, dcorr(im,REF), clip_prompt(im,TARGET)))
cell=300; grid=Image.new("RGB",(len(panels)*cell+(len(panels)+1)*6, cell+34),"white"); d=ImageDraw.Draw(grid)
for j,(name,im,dc,cl) in enumerate(panels):
    x=6+j*(cell+6); grid.paste(im.resize((cell,cell)),(x,4))
    lab=name if dc is None else f"{name} d={dc:.2f} clip={cl:.2f}"; d.text((x+4,cell+10),lab[:36],fill="black")
display(grid)
print("GO if a strength keeps depth-corr high-ish (~0.7-0.9 = ref layout) AND clip-to-prompt near the no-control")
print("value (content followed the prompt). Spike found strength=0.3 the sweet spot; confirm + bake as default.")